# Encode categorical variables

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("../Data/Telco_Customer_Churn_processed.csv")

In [3]:
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [4]:
# Multi-category columns → one-hot encode

categorical_cols = ["gender", "MultipleLines", "InternetService", "OnlineSecurity",
"OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
"StreamingMovies", "Contract", "PaymentMethod"]

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

## Buiseness relevant features

In [11]:
# Tenure groups 
df_encoded["is_new_customer"] = (df["tenure"] <= 6).astype(int)

# Number of subscribed add-on services 
service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
"TechSupport", "StreamingTV", "StreamingMovies"]
df_encoded["num_services"] = df[service_cols].apply(
lambda row: sum(1 for v in row if v == "Yes"), axis=1
)

# Average revenue per month of tenure 
df_encoded["avg_monthly_spend"] = df["TotalCharges"] / df["tenure"].replace(0, 1)

# High-value customer (top 25% by monthly charges)
threshold = df["MonthlyCharges"].quantile(0.75)
df_encoded["is_high_value"] = (df["MonthlyCharges"] > threshold).astype(int)

# Contract risk flag — month-to-month
df_encoded["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype(int)



## Handling class imbalance

In [12]:
from sklearn.model_selection import train_test_split
X = df_encoded.drop(columns=["customerID", "Churn"])
y = df_encoded["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
df_encoded.to_csv("../Data/Telco_Customer_Churn_Features.csv", index=False)